# Evaluating AI Outputs — the cheat sheet

**AI Reality Check: Decide, Defend, Deliver** · 30 September 2026 · Monique Marins

You just voted on five AI-generated insights. Two were wrong. This notebook is the part you
take home: the checks, running on the workshop data, written so you can point them at your
own.

**How to use it**

1. Run it top to bottom. It reads `ecommerce_dataset.csv` and `business_context.csv` from the repo's `data/` folder.
2. Sections 1–5 reproduce the session with the numbers live.
3. **Section 7 is the one that matters** — the same checks against your own data and your own
   answer key.

Nothing here calls an AI. That is the point: the detection is arithmetic, so it gives the
same answer every time and anyone can redo it.

---

## The one-page version

| | What it is | Catches | Misses |
|---|---|---|---|
| **1. Fix the prompt** | Make the model fill fixed fields: metric, window, baseline, evidence | Cheapest to try. Makes everything else checkable | Cannot tell you when it failed. Same prompt, different answer tomorrow |
| **2. LLM as judge** | A second model scores the first against a rubric | Invented causes, misleading framing | Cannot see what was left out. Can be expensive |
| **3. Statistical checks** | Recompute in code, compare against a list of what was really there | Wrong numbers, every time. **Omissions.** Cheap to run | Needs a human to build. Only tests what you thought of |

**The four questions behind all of it**

1. What counts as normal? → your baseline
2. How far off before you call it a problem? → your cut-off
3. What was really there? → your answer key
4. What does a miss cost, against what a check costs? → your threshold decision


## Setup

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

DATA = Path("../data")   # the repo's data folder; change if your CSVs live elsewhere

daily   = pd.read_csv(DATA / "ecommerce_dataset.csv", parse_dates=["date"])
context = pd.read_csv(DATA / "business_context.csv",  parse_dates=["date"])

df = daily.merge(context, on="date", how="left")
df["dow"] = df["date"].dt.dayofweek
print(f"{len(df)} days, {df.date.min():%d %b %Y} to {df.date.max():%d %b %Y}")
df.head()

---

## 1. Fix the prompt

Before any of the checks below, make the output *checkable*. Insight 04 in the session
("likely system failure") was only possible because nothing forced the model to say where
the cause came from. A free-text sentence cannot be verified without redoing the analysis.

### The prompt that produced the five insights

In [ ]:
WEAK_PROMPT = """
Analyse this dataset and give me the 5 most important insights for a business dashboard.
"""
print(WEAK_PROMPT)

Nothing here asks for a baseline, a window, or evidence — so the model invents all
three, fluently. Every failure the room voted on came out of that one sentence.

### The same request, written so the answer can be checked

In [ ]:
STRONG_PROMPT = """
You are analysing daily e-commerce metrics.

FILES
  ecommerce_dataset.csv  - daily metrics
  business_context.csv   - holidays, campaigns, product releases, expected multipliers

TASK
  Return at most 5 findings, as a JSON list. One object per finding, these fields only:

  metric           the column name, exactly as it appears in the file
  window           start and end date, YYYY-MM-DD
  observed         the value over that window
  baseline         the value you compared against
  baseline_method  how you computed the baseline, in one line
  change           observed vs baseline, as a multiple or a percentage
  evidence         file and column that supports a cause, or null
  cause            only if evidence is not null, otherwise null
  confidence       "verified" | "probable" | "unclear"

RULES
  1  Compare against the same weekday, never the overall average.
  2  Apply expected_revenue_multiplier from business_context.csv before
     calling anything unusual.
  3  When volume changed, report a rate, not a count.
  4  If you cannot compute a baseline from the files, return null. Do not estimate.
  5  Never state a cause with evidence = null.
  6  State the direction against the BASELINE, not against the previous day.
"""
print(STRONG_PROMPT)

**Every rule exists because of a failure the room just voted on.** That is what makes
this a contract rather than prompt-engineering folklore.

| Rule | The failure it prevents |
|---|---|
| 1 — same weekday | Weekends run ~31% below weekdays, so half the calendar looks broken |
| 2 — apply the multiplier | Black Friday flagged as a critical incident |
| 3 — rate, not count | Tickets "4.6x normal" on Black Friday; per 1,000 orders they were normal |
| 4 — null, never estimate | A baseline nobody can trace, like the untraceable "$18,000" |
| 5 — no cause without evidence | "likely system failure" when the file said `is_release_day` |
| 6 — direction vs baseline | "Revenue dropped 16%" on the day revenue rose 280% |

### Check the shape automatically

A contract is only worth having if something enforces it. This runs in milliseconds and
needs no model.

In [ ]:
REQUIRED = ["metric", "window", "observed", "baseline", "baseline_method",
            "change", "evidence", "cause", "confidence"]


def check_shape(insight):
    """Validate one AI finding against the output contract. Returns a list of problems."""
    problems = []

    for f in REQUIRED:
        if f not in insight:
            problems.append(f"missing field: {f}")

    if insight.get("cause") and not insight.get("evidence"):
        problems.append("cause claimed with evidence = null  (rule 5)")

    if insight.get("baseline") is None:
        problems.append("baseline is null - the change cannot be checked  (rule 4)")

    if not insight.get("baseline_method"):
        problems.append("no baseline_method - nobody can reproduce the comparison")

    if insight.get("confidence") not in {"verified", "probable", "unclear", None}:
        problems.append("confidence not one of the three allowed values")

    for f in ("observed", "baseline"):
        v = insight.get(f)
        if v is not None and not isinstance(v, (int, float)):
            problems.append(f"{f} should be a number, got {type(v).__name__}")

    return problems


# --- the output as the AI first gave it (insight 04) ---
bad = {"metric": "support_tickets", "window": "2025-10-20",
       "observed": 56, "baseline": None, "baseline_method": None,
       "change": "3x normal", "evidence": None,
       "cause": "likely system failure", "confidence": "probable"}

# --- the same finding, written to the contract ---
good = {"metric": "support_tickets", "window": "2025-10-20 to 2025-10-22",
        "observed": 63, "baseline": 18,
        "baseline_method": "median of quiet days (no campaign), daily average",
        "change": "3.5x", "evidence": "business_context.csv: is_release_day",
        "cause": "release on 2025-10-20", "confidence": "probable"}

for name, ins in [("AS GENERATED", bad), ("TO THE CONTRACT", good)]:
    issues = check_shape(ins)
    print(f"{name}: {'PASS' if not issues else str(len(issues)) + ' problem(s)'}")
    for p in issues:
        print(f"    - {p}")
    print()

The bad version fails on three counts, and one of them — *cause claimed with
evidence = null* — is exactly the mistake the room caught by eye. Now a machine catches it,
on every output, for free.

> **A prompt is not a fix, it is a reduction.** The same prompt can give you a different
> answer tomorrow, and a model upgrade can undo your work silently. That is why sections 2
> to 6 exist.

---

## 2. What counts as normal?

Three answers, worst to best. Each is a *baseline*: the number you compare today against.
Only the last one can tell an expected peak from a real problem.

In [ ]:
def baseline_weekday(df, metric):
    """Compare each day to the median of the same weekday, quiet days only.

    'Quiet' = no campaign running. Letting campaign days define normal is how
    Black Friday ends up inside your definition of a typical day.
    """
    quiet = df[df["expected_revenue_multiplier"] == 1]
    med   = quiet.groupby("dow")[metric].median()
    return df["dow"].map(med)


def baseline_expected(df, metric):
    """Weekday median x the campaign multiplier from the context file."""
    return baseline_weekday(df, metric) * df["expected_revenue_multiplier"]


# The naive version first: distance from one overall average, in standard deviations.
z = (df.revenue_usd - df.revenue_usd.mean()) / df.revenue_usd.std()
print("z-score against one flat average   (flag if |z| > 2)")
for label, d_ in [("5 Nov, real outage", "2025-11-05"),
                  ("28 Nov, Black Friday", "2025-11-28")]:
    print(f"   {label:24s} z = {z[df.date == d_].iloc[0]:6.2f}")

print()
for name, fn in [("same weekday", baseline_weekday),
                 ("weekday x calendar", baseline_expected)]:
    b = fn(df, "revenue_usd")
    a = df.revenue_usd[df.date == "2025-11-05"].iloc[0] / b[df.date == "2025-11-05"].iloc[0]
    c = df.revenue_usd[df.date == "2025-11-28"].iloc[0] / b[df.date == "2025-11-28"].iloc[0]
    print(f"{name:20s} 5 Nov {a:5.2f}      28 Nov {c:5.2f}")

**Read the two columns.** 5 November was a genuine outage, so you want that number far from
normal. 28 November was Black Friday, so you want that one to look *ordinary*, because nothing
was wrong.

The z-score gets both wrong. It scores the outage at −1.78, inside the bar, because Black
Friday and Christmas sit in the same pile it measures the spread of — they stretch what counts
as normal until an 87% collapse no longer stands out. The method did not fail; the reference
set did.

Only the last baseline gets both right.

---

## 3. How far off before you call it a problem?

Do not pick the cut-off by hand. Measure how much ordinary days already wobble, and flag
anything well outside that.

In [ ]:
def cut_off(df, metric, baseline_fn, n_sd=3, exclude=None):
    """Derive a flagging range from the spread of ordinary days.

    exclude: dates you already know were incidents. Leave them in and they widen the
             very spread you measure against, which is how an incident hides itself.
    """
    ratio    = df[metric] / baseline_fn(df, metric)
    ordinary = df["expected_revenue_multiplier"] == 1
    if exclude is not None:
        ordinary &= ~df["date"].isin(pd.to_datetime(exclude))
    r = ratio[ordinary]
    lo, hi = r.mean() - n_sd * r.std(), r.mean() + n_sd * r.std()
    return round(max(lo, 0), 2), round(hi, 2)      # a ratio cannot go below zero


# The 5 Nov outage, plus the late-December slump (section 5 shows why that data is suspect).
KNOWN_INCIDENTS = ["2025-11-05"] + list(
    pd.date_range("2025-12-26", "2025-12-31").strftime("%Y-%m-%d"))

lo, hi = cut_off(df, "revenue_usd", baseline_weekday, n_sd=3, exclude=KNOWN_INCIDENTS)
print(f"Ordinary days sit between {lo} and {hi} of their weekday median.")
print("Anything outside that is worth a look.")

---

## 4. Detect

One function. Point it at any metric and any baseline.

In [ ]:
def detect(df, metric, baseline_fn, lo, hi):
    """Return the days where metric / baseline falls outside [lo, hi]."""
    out = df[["date", metric]].copy()
    out["expected"] = baseline_fn(df, metric).round(0)
    out["ratio"]    = (out[metric] / baseline_fn(df, metric)).round(2)
    return out[(out.ratio < lo) | (out.ratio > hi)].reset_index(drop=True)


print("A - same weekday only")
print(detect(df, "revenue_usd", baseline_weekday, lo, hi).to_string(index=False))

print()
print("B - weekday x campaign calendar")
print(detect(df, "revenue_usd", baseline_expected, lo, hi).to_string(index=False))

Method A flags the real outage **and** Black Friday, a Christmas-campaign day (18 Dec), Christmas Eve and Christmas Day.
Method B flags the outage and leaves the holidays alone — same cut-off, the only difference is
that it knows what was already planned.

(Method B also flags 30 November. That one is honest: the context file gives Black Friday week
a 3.0x multiplier that the actual revenue never reached. A baseline is only as good as the
calendar behind it.)

---

## 5. Score it against an answer key

Almost nobody builds this, and it is the only thing that tells you whether a change actually
improved anything.

In [ ]:
def score(flagged_dates, answer_key):
    """Compare what you found against what was really there."""
    found, truth = set(pd.to_datetime(flagged_dates)), set(pd.to_datetime(answer_key))
    tp, fp, fn = found & truth, found - truth, truth - found
    return {
        "true positives":  sorted(d.strftime("%d %b") for d in tp),
        "false positives": sorted(d.strftime("%d %b") for d in fp),
        "false negatives": sorted(d.strftime("%d %b") for d in fn),
        "precision": round(len(tp) / len(found), 2) if found else float("nan"),
        "recall":    round(len(tp) / len(truth), 2) if truth else float("nan"),
    }


ANSWER_KEY = ["2025-11-05"]          # the one genuine revenue incident

for label, fn_ in [("A  same weekday      ", baseline_weekday),
                   ("B  weekday x calendar", baseline_expected)]:
    hits = detect(df, "revenue_usd", fn_, lo, hi)["date"]
    s = score(hits, ANSWER_KEY)
    print(f"{label}  recall {s['recall']}   precision {s['precision']}"
          f"   missed: {s['false negatives'] or 'none'}"
          f"   false alarms: {len(s['false positives'])}")

**Recall** = of the real problems, how many you found.
**Precision** = of your alarms, how many were real.

You do not optimise both. Which one matters is a cost question, not a statistical one — that
is section 6.

> **A perfect score only covers what is in the answer key.** Late December looked odd, so
> someone looked. Run the cell below.

In [ ]:
dupes = df[df.duplicated(subset=["revenue_usd", "customer_churn_rate"], keep=False)]
print(dupes[["date", "revenue_usd", "orders_count", "customer_churn_rate"]].to_string(index=False)
      if len(dupes) else "No duplicated rows found.")

Two different days cannot produce identical revenue *and* identical churn to the decimal.
The pipeline had copied a day. No detector above flagged it and the answer key did not list
it. Nobody reading the insights would ever have found this.

---

## 6. What should the threshold be? (interactive)

To see a real trade-off we need a metric with more than one incident in it. Use support tickets
as a **rate** — tickets per 1,000 orders — against the four incidents we know about.

The question in plain words: *how many times normal before you raise an alarm?*

Move the sliders. Set the bar low and you read noise; raise it and you start missing real
things. Then drag `cost_of_miss` down and `cost_of_check` up, and watch the best answer flip.

In [ ]:
df["tickets_per_1k"] = df.support_tickets / df.orders_count * 1000

INCIDENTS = ["2025-10-20", "2025-10-21", "2025-10-22", "2025-11-05"]

ratio_t1k = df.tickets_per_1k / baseline_weekday(df, "tickets_per_1k")


def sweep(times_normal=2.0, cost_of_miss=16500, cost_of_check=25):
    hits = df.date[ratio_t1k > times_normal]
    s = score(hits, INCIDENTS)
    missed, alarms = len(s["false negatives"]), len(s["false positives"])
    total = missed * cost_of_miss + len(hits) * cost_of_check
    print(f"alarm when tickets/1k orders is more than {times_normal:.1f}x normal")
    print(f"   {len(hits)} days flagged this quarter")
    print(f"   caught {len(s['true positives'])}/{len(INCIDENTS)}     missed {missed}     false alarms {alarms}")
    print(f"   recall {s['recall']}     precision {s['precision']}")
    print(f"   cost of the misses   ${missed * cost_of_miss:>8,}")
    print(f"   cost of the checks   ${len(hits) * cost_of_check:>8,}")
    print(f"   TOTAL                ${total:>8,}")
    if s["false negatives"]:
        print(f"   -> you just missed: {', '.join(s['false negatives'])}")


try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(sweep,
             times_normal=FloatSlider(value=2, min=1, max=6, step=0.1, description="x normal"),
             cost_of_miss=IntSlider(value=16500, min=0, max=50000, step=500, description="cost/miss"),
             cost_of_check=IntSlider(value=25, min=0, max=2000, step=25, description="cost/check"))
except ImportError:
    print("ipywidgets not installed - showing a table instead.")
    print("pip install ipywidgets  for the sliders")
    print()
    for t in [1.2, 2.0, 3.2, 5.0]:
        sweep(t)
        print()

**The asymmetry is the argument.** A missed incident cost around \$16,500 in a day. A false
alarm costs a few minutes of someone looking. At that ratio you never raise the bar high enough
to miss one — *for this metric*. Something expensive to check and cheap to get wrong runs a
higher bar instead.

Set `cost_of_miss` to a few hundred and `cost_of_check` to a couple of thousand, and 5x becomes
the cheapest setting. That is the whole point
in one slider: the right threshold is a business number, not a statistical one.

---

## 7. Your turn

Everything above works on any table with a date column and a number. Fill in the four blanks.

In [ ]:
# ---------------------------------------------------------------
# 1. YOUR DATA - any dataframe with a date column and a metric
# ---------------------------------------------------------------
my_df = df.copy()          # <- replace with your own
                           #    needs a `dow` column, and `expected_revenue_multiplier` (1 if you have no calendar)

# ---------------------------------------------------------------
# 2. YOUR BASELINE - what counts as normal here?
#    Tip: for anything that scales with volume, use a RATE, not a count.
#    Black Friday tickets were 4.6x the quiet-day median - but tickets per 1,000
#    orders were completely normal. Same day, one division.
# ---------------------------------------------------------------
my_df["tickets_per_1k_orders"] = my_df.support_tickets / my_df.orders_count * 1000
MY_METRIC   = "tickets_per_1k_orders"
my_baseline = baseline_weekday

# ---------------------------------------------------------------
# 3. YOUR ANSWER KEY - dates you know were genuinely wrong.
#    Write this BEFORE you look at the results. An answer key written
#    afterwards is not a test, it is a justification.
# ---------------------------------------------------------------
MY_ANSWER_KEY = INCIDENTS   # <- replace with your own list

# ---------------------------------------------------------------
# 4. RUN
# ---------------------------------------------------------------
my_lo, my_hi = cut_off(my_df, MY_METRIC, my_baseline, n_sd=3, exclude=MY_ANSWER_KEY)
my_hits      = detect(my_df, MY_METRIC, my_baseline, my_lo, my_hi)

print(f"cut-off {my_lo} to {my_hi}")
print(my_hits.to_string(index=False))
print()
for k, v in score(my_hits["date"], MY_ANSWER_KEY).items():
    print(f"{k:>16}: {v}")

Try it with `support_tickets` as a raw count instead of the rate. The count flags Black
Friday; the rate does not. Sometimes you do not need a smarter method or a context file — you
need the right denominator.

---

## Before you ship anything

- [ ] **Is every number in the output traceable to a value in the data?** If not, you cannot check it.
- [ ] **Is a cause claimed?** Then a source must be cited. No source, no cause.
- [ ] **Is it a count or a rate?** Counts rise with volume. Rates do not.
- [ ] **Did you write the answer key before you saw the results?**
- [ ] **What does one miss cost?** That number sets your threshold, not a p-value.
- [ ] **Would a reader without the data misread the direction?** "Revenue dropped 16%" on the biggest sales day of the year is technically defensible and completely misleading.
- [ ] **What would the check never have mentioned?** Only an answer key finds omissions.

---

*Questions after the session — come find me.*
*Monique Marins · Applied AI Manager, Trustpilot*